In [1]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("RutaVivaDataPipeline")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"Sesión de Spark iniciada correctamente. Versión: {spark.version}")

Sesión de Spark iniciada correctamente. Versión: 3.5.0


In [6]:
file_path = "file:///home/jovyan/work/services/data-pipeline/app/data/historical/2015/01_clean/part-00000-6822f9ca-b7a3-4ae1-a575-3d31c992365c-c000.csv"

df2015_1 = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ",")
    .load(file_path))

In [7]:
file_path = "file:///home/jovyan/work/services/data-pipeline/app/data/historical/2016/01_clean/part-00000-524ca9ce-2356-4a06-9179-641e89563a8e-c000.csv"

df2016_1 = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ",")
    .load(file_path))

In [8]:
file_path = "file:///home/jovyan/work/services/data-pipeline/app/data/historical/2016/02_clean/part-00000-449887de-0e1a-4406-84d5-153c7fce5ffb-c000.csv"

df2016_2 = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ",")
    .load(file_path))

In [10]:
file_path = "file:///home/jovyan/work/services/data-pipeline/app/data/historical/2016/03_clean/part-00000-4984cc5f-645e-466e-b54a-82af1dd64991-c000.csv"

df2016_3 = (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("delimiter", ",")
    .load(file_path))

In [11]:
historical_data = df2015_1.union(df2016_1).union(df2016_2).union(df2016_3)

historical_data.show()

+--------------------+---------------------+------------------+------------------+------------------+------------------+-----------+-----+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|  Pickup_longitude|   Pickup_latitude| Dropoff_longitude|  dropoff_latitude|fare_amount|extra|total_amount|
+--------------------+---------------------+------------------+------------------+------------------+------------------+-----------+-----+------------+
| 2015-01-15 19:05:39|  2015-01-15 19:23:42|  -73.993896484375|  40.7501106262207|-73.97478485107422| 40.75061798095703|       12.0|  1.0|       17.05|
| 2015-01-10 20:33:38|  2015-01-10 20:53:28|-74.00164794921875|  40.7242431640625|-73.99441528320312| 40.75910949707031|       14.5|  0.5|        17.8|
| 2015-01-10 20:33:38|  2015-01-10 20:43:41|-73.96334075927734| 40.80278778076172|-73.95182037353516| 40.82441329956055|        9.5|  0.5|        10.8|
| 2015-01-10 20:33:39|  2015-01-10 20:35:31|-74.00908660888672| 40.71381759643555|-74.00

In [12]:
from pyspark.sql.functions import col, date_format

# 1. Extrae el formato año-mes ("YYYY-MM"), saca los valores únicos y ordénalos
df_meses_unicos = (
    historical_data.select(date_format(col("tpep_pickup_datetime"), "yyyy-MM").alias("mes"))
    .distinct()
    .orderBy("mes")
)

df_meses_unicos.show()

+-------+
|    mes|
+-------+
|2015-01|
|2016-01|
|2016-02|
|2016-03|
+-------+



In [13]:
# Guardar el DataFrame en formato Parquet
historical_data.write.mode("overwrite").parquet(
    "data/historical/taxi_data_parquet"
)